<center>
    <font size="5"> Sieci neuronowe i uczenie głębokie<br/>
        <small><em>Studia stacjonarne II stopnia 2025/2026</em><br/>Kierunek: Matematyka stosowana<br>Specjalność: Analityka danych</small>
    </font>
</center>
<br>



# Laboratorium nr 4: Konwolucyjne sieci neuronowe: Klasyfikacji obrazów

## Import bibliotek

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
print('Numpy version:', np.__version__)
print('Tensorflow version:', tf.__version__)

## Baza MNIST

- Baza MNIST zawiera zbiór trenujący składający się z 60,000 przykładów skanów ręcznie pisanych cyfr od 0 do 9 (problem klasyfikacyjny z 10 klasami).
- Zbiór testowy zawiera 10,000 przykładów.
- Każdy obraz ma rozmiar 28x28 pikseli. Stanowią one 28 * 28 = 784 wejść do sieci.
- W zagadnieniach rozpoznawania obrazów baza MNIST pełni rolę swoistego problemu `Hello world`.

#### Pobieranie bazy MNIST
- Bazę MNIST można pobrać bespośrednio ze strony http://yann.lecun.com/exdb/mnist/
- Najwygodniej jednak jest użyć bazy MNIST z wykorzystaniem biblioteki Keras

In [ ]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()
print(train_images.shape, train_labels.shape, test_images.shape, test_labels.shape)
train_images = np.expand_dims(train_images, axis=-1)/255.
train_labels = np.int64(train_labels)
test_images = np.expand_dims(test_images, axis=-1)/255.
test_labels = np.int64(test_labels)
print(train_images.shape, train_labels.shape, test_images.shape, test_labels.shape)

Przykładowe cyfry:

In [ ]:
plt.figure(figsize=(10,10))
random_inds = np.random.choice(60000, 36)
for i in range(36):
    plt.subplot(6,6,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    image_ind = random_inds[i]
    plt.imshow(np.squeeze(train_images[image_ind]), cmap=plt.cm.binary)
    plt.xlabel(train_labels[image_ind])

## Sieć wielowarstwowa

### Tworzenie modelu

#### Ćwiczenie 1:
Sieć pełnych połączeń
- Stwórz w bibliotece keras model klasycznej sieci wielowarstwowej z dwiema warstwami ukrytymi.
- Dla pierwszej warstwy ustaw liczbę 128 neuronów
- Dla drugie warstwy liczbę neuronów odpowiedającą liczbie klas.
- Dobierz odpowiednie funkcje aktywacji.

In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, Flatten

In [ ]:
def fc_model():
    x = Input(shape=(28,28,1))
    xf = Flatten()(x)
    #TODO:
    # dodaj pierwszą warstwę ukrytą
    # dodaj drugą warstwę ukrytą
    return Model(inputs=x, outputs=y)

model = fc_model()
model.summary()

### Kompilacja

#### Ćwiczenie 2:
Skompiluj model ustawiając:
- funkcje straty: `sparse_categorical_crossentropy`
- algorytm optymalizacji: np. `SGD`
- metryki monitorujące proces uczenia: `accuracy`

In [ ]:
model.compile()

### Trenowanie
Trenowanie z wykorzystaniem podziału na dane uczące i walidacyjne, oraz `ModelCheckpoint`, czylu zapisywanie najlepszego modelu

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

In [ ]:
# aby zapisać model na Google Drive podaj odpowiednią ścieżkę
cb = [ModelCheckpoint('model_fc_mnist.keras', monitor='val_accuracy', save_best_only=True)]

In [ ]:
BATCH_SIZE = 64
EPOCHS = 20

In [ ]:
model.fit(train_images, train_labels, batch_size=BATCH_SIZE, epochs=EPOCHS, validation_split=0.2, callbacks=cb)

### Ewaluacja zbioru testowego

In [ ]:
model.evaluate(test_images, test_labels)

Model z pliku

In [ ]:
fmodel = tf.keras.models.load_model('model_fc_mnist.keras')
fmodel.evaluate(test_images, test_labels)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
predictions = fmodel.predict(test_images)
predictions = np.argmax(predictions, axis=1)
cm = confusion_matrix(test_labels, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()

## Najlepsze wyniki dla bazy MNIST
http://rodrigob.github.io/are_we_there_yet/build/classification_datasets_results.html#4d4e495354

## Sieć konwolucyjna

### Prosty model

#### Ćwiczenie 3:

- Stwórz w bibliotece keras model prostej sieci konwolucyjnej z 1 warstwą conwolucyjną, 1 warstwą MaxPooling i warstwami klasyfikacyjnymi jak w poprzednim modelu.

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dropout

In [ ]:
def conv_model():
    x = Input(shape=(28,28,1))
    #TODO:
    return Model(inputs=x, outputs=y)

model = conv_model()
model.summary()

### Kompilacja i trenowanie

#### Ćwiczenie 4:
- Skompiluj i wytrenuj model ustawiając parametry jak przy sieci wielowarstwowej.

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
cb = [ModelCheckpoint('model_conv_mnist.keras', monitor='val_accuracy', save_best_only=True)]

In [ ]:
model.fit(train_images, train_labels, batch_size=BATCH_SIZE, epochs=EPOCHS, validation_split=0.2, callbacks=cb)

### Ewaluacja zbioru testowego

In [ ]:
model.evaluate(test_images, test_labels)

Model z pliku

In [ ]:
fmodel = tf.keras.models.load_model('model_conv_mnist.keras')
fmodel.evaluate(test_images, test_labels)

## Ćwiczenie 5:
Zdefiniuj, wytrenuj i przetestuj model sieci konwolucyjnej jak na rysunku:
![convnet](http://torus.uck.pk.edu.pl/~amarsz/images/zmum/cw5.PNG)

In [ ]:

def conv_model():
    x = Input(shape=(28,28,1))
    #TODO:
    return Model(inputs=x, outputs=y)

model = conv_model()
model.summary()

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
cb = [ModelCheckpoint('model_conv2_mnist.keras', monitor='val_accuracy', save_best_only=True)]

In [ ]:
model.fit(train_images, train_labels, batch_size=BATCH_SIZE, epochs=20, validation_split=0.2, callbacks=cb)

In [ ]:
fmodel = tf.keras.models.load_model('model_conv2_mnist.keras')
fmodel.evaluate(test_images, test_labels)

In [ ]:
predictions = fmodel.predict(test_images)
predictions = np.argmax(predictions, axis=1)
cm = confusion_matrix(test_labels, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()

## Wizualizacja Predykcji
Za pomocą poniższych funkcji zwizualizuj wyniki predykcj modelu z ćwiczenia 5 na zbiorze testowym.

In [ ]:
def plot_image(i, predictions_array, true_label, img):
    predictions_array, true_label, img = predictions_array[i], true_label[i], img[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])

    plt.imshow(np.squeeze(img), cmap=plt.cm.binary)

    predicted_label = np.argmax(predictions_array)
    if predicted_label == true_label:
        color = 'blue'
    else:
        color = 'red'

    plt.xlabel("{} {:2.0f}% ({})".format(predicted_label,
                                         100*np.max(predictions_array),
                                         true_label), color=color)

def plot_value_array(i, predictions_array, true_label):
    predictions_array, true_label = predictions_array[i], true_label[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    thisplot = plt.bar(range(10), predictions_array, color="#777777")
    plt.ylim([0, 1])
    predicted_label = np.argmax(predictions_array)

    thisplot[predicted_label].set_color('red')
    thisplot[true_label].set_color('blue')

In [ ]:
predictions = fmodel.predict(test_images)

In [ ]:
num_rows = 5
num_cols = 4
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, predictions, test_labels, test_images)
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, predictions, test_labels)

## Wizualizacja filtrów i map cech
Zwizualizuj filtry w pierwszej warstwie konwolucyjnej oraz mapy cech powstałe po pierwszej warstwie konwolucyjnej